In [3]:
# Clear all variables
%reset -f
%cd /home/zwei1/Artem/git_book/handbook_chapter/sfa-python/sfa/inefficiency_predictors/

/home/zwei1/Artem/git_book/handbook_chapter/sfa-python/sfa/inefficiency_predictors


In [ ]:
from util import JLMS_panel_technical_inefficiency_scores
import numpy as np
import matplotlib.pyplot as plt 

In [ ]:
#load parameters


theta = np.array([
    5.7630,
    0.1422,
    0.1170,
    0.0720,
    0.1851,
    0.5150,
    -0.0191,
    0.1561,
    0.0996,
    -0.0355,
    -0.1337,
    -0.1617,
    -0.1609,
    -0.0293,
    0.6475,
    2.0451
])

print(theta)

gamma = np.array([
4.691359544478445054e-01,
2.449463444476085439e-01,
4.738861695923803768e-02,
1.416673024522306368e-01,
2.191707371168801211e-01,
9.297518610871904343e-01,
7.927239502606412413e-01,
1.393802658718614085e+00,
3.252880911850812573e-02,
-4.430338547558972384e-01,
-9.527848585280578320e-01,
1.232980848191408789e-02,
-1.114131598725027317e+00,
2.370215935915223338e-01,
7.045769769525236370e-01
])
gamma

[ 5.763   0.1422  0.117   0.072   0.1851  0.515  -0.0191  0.1561  0.0996
 -0.0355 -0.1337 -0.1617 -0.1609 -0.0293  0.6475  2.0451]


array([ 0.46913595,  0.24494634,  0.04738862,  0.1416673 ,  0.21917074,
        0.92975186,  0.79272395,  1.39380266,  0.03252881, -0.44303385,
       -0.95278486,  0.01232981, -1.1141316 ,  0.23702159,  0.70457698])

In [ ]:
ricefarm = pd.read_csv(r"ricefarm.csv")

ricefarm = ricefarm.iloc[:, [0, 1, 3, 5, 6, 7, 8, 14, 16]]

# Create ID dummies
dar = np.round(ricefarm["ID"] / 100000).astype(int)
id_dummies = pd.get_dummies(dar, prefix="DR").astype(int)
ricefarm = pd.concat([ricefarm, id_dummies.iloc[:, :-1]], axis=1)

# Create rice variety dummies
rice_dummies = pd.get_dummies(ricefarm.iloc[:, 2], prefix="VAR").astype(int)
ricefarm = pd.concat([ricefarm, rice_dummies.iloc[:, 1:]], axis=1)

# Recode TSP as logs and zeros
ricefarm[ricefarm.columns[5]] = np.where(
    ricefarm.iloc[:, 5] > 0, np.log(ricefarm.iloc[:, 5]), 0
)

# Convert pesticide usage to a dummy
ricefarm[ricefarm.columns[6]] = (ricefarm.iloc[:, 6] != 0).astype(int)

# Reorder the data
ricefarm = ricefarm.iloc[:, [8, 3, 4, 5, 7, 1, 6, 14, 15, 9, 10, 11, 12, 13]]

y = np.log(ricefarm.iloc[:, 0].values)
X = ricefarm.iloc[:, 1:].values

# Log covariates
X[:, 0:2] = np.log(X[:, 0:2])
X[:, 3] = np.log(X[:, 3])
X[:, 4] = np.log(X[:, 4])

# Cross sections of  y and X
N = 171
T = 6
y_t = []
X_t = []
for t in range(T):
    if t == 0:
        y_t.append(y[:N])
        X_t.append(X[:N, :])
    else:
        y_t.append(y[t * N : (t + 1) * N])
        X_t.append(X[t * N : (t + 1) * N, :])

# Import the model parameters from exercise 5.8
# theta = np.loadtxt("exercise_4_8_theta_python.txt", delimiter=",")

# Estimation of technical inefficiencies
# Get idx of percentile firms
percetile_firm_idx = np.round(np.arange(0.05, 1, 0.1) * N).astype(int)

# JLMS formula technical inefficiencies
JLMS_u_hat = JLMS_panel_technical_inefficiency_scores(theta=theta, y=y_t, X=X_t)
# Sort technical inefficiency scores by t=1 values
sort_idx = JLMS_u_hat[:, 0].argsort()
sorted_t1_JLMS_u_hat = JLMS_u_hat[sort_idx]
# Mean technical inefficiency for each firms over all t
JLMS_mean = np.mean(JLMS_u_hat, axis=1)
sorted_t1_JLMS_mean = JLMS_mean[sort_idx]
# Standard deviation of technical inefficiency for each firm over all t
JLMS_std = np.std(JLMS_u_hat, axis=1)
sorted_t1_JLMS_std = JLMS_std[sort_idx]

JLMS_u_hat_for_table = sorted_t1_JLMS_u_hat[percetile_firm_idx-1, :]
JLMS_std_u_hat_for_table = sorted_t1_JLMS_std[percetile_firm_idx-1]
JLMS_mean_u_hat_for_table = sorted_t1_JLMS_mean[percetile_firm_idx-1]

# Tabulate the technical inefficiency scores
JLMS_technical_inefficiency_results = pd.DataFrame(
    columns=[
        "Rank",
        "Fractile",
        "Std(u)",
        "Mean(u)",
        "t=1",
        "t=2",
        "t=3",
        "t=4",
        "t=5",
        "t=6",
    ]
)
JLMS_technical_inefficiency_results["Rank"] = percetile_firm_idx
JLMS_technical_inefficiency_results["Fractile"] = np.arange(0.05, 1, 0.1)
JLMS_technical_inefficiency_results["Std(u)"] = JLMS_std_u_hat_for_table
JLMS_technical_inefficiency_results["Mean(u)"] = JLMS_mean_u_hat_for_table
JLMS_technical_inefficiency_results.iloc[:, -6:] = JLMS_u_hat_for_table

# Compute averages
average_JLMS = pd.DataFrame(
    np.concatenate(
        [
            np.mean(JLMS_std_u_hat_for_table).reshape(-1, 1).T,
            np.mean(JLMS_mean_u_hat_for_table).reshape(-1, 1).T,
            np.mean(JLMS_u_hat_for_table, axis=0).reshape(-1, 1).T,
        ],
        axis=1,
    ),
    columns=["Std(u)", "Mean(u)", "t=1", "t=2", "t=3", "t=4", "t=5", "t=6"],
    index=["Average"],
)

print("JLMS Scores")
display(JLMS_technical_inefficiency_results)
display(average_JLMS)